# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
!py -m pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ---------------------------------------- 952.8/952.8 kB 13.8 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [elastic-transport]
   ---------- ----------------------------- 1/4 [elastic-transport]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------------------- ------------------- 2/4 [elasticsearch]
   -------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [26]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="retiver_chatbot", #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [27]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [28]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")
    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [29]:
from langchain_openai import ChatOpenAI

documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="retiver_chatbot",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-6-luna",
    temperature=1,
)

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [30]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is the NASA sales team, and what does it do?', 'Who are the members of NASA’s sales team?', 'What does “NASA sales team” refer to?']


---- Answer ----
content='The NASA sales team is the **North America–South America** region, covering the United States, Canada, Mexico, Central America, and South America. It is led by two Area Vice-Presidents: **Laura Martinez** for North America and **Gary Johnson** for South America.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 458, 'total_tokens': 563, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 38, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-eaaa558e-f7c9-4676-94a3-b200698e05ed-0' usage_metadata={'input_tokens': 458, 'output_tokens': 105, 'total_tokens': 563}


In [31]:
ans = chain.invoke("How many days a week must employees be in the office, and how has that changed over time?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is the required number of in-office days per week for employees, and how has the policy evolved?', 'How often must employees work from the office each week, and what changes have been made to that requirement over time?', 'What are the current and past expectations for employees’ weekly office attendance?']


---- Answer ----
content='Starting in May 2022, eligible employees were required to work in the office **two days a week**. From **May 1, 2023**, the requirement increased to **three days a week**, with **two days remote**—one additional in-office day per week.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 230, 'total_tokens': 348, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 51, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-2213557c-e1e0-4dd3-8cc5-2ed78c6e1a62-0' usage_metadata={'input_tokens': 230, 'output_tokens': 118, 'total_tokens': 348}


In [32]:
ans = chain.invoke("How many vacation days do I get per year?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['How much annual vacation leave am I entitled to?', 'What is my yearly vacation-day allowance?', 'How many paid time-off days do employees receive each year?']


---- Answer ----
content='The policy doesn’t specify an exact number: it lists **[Y days] per year** for full-time employees. Part-time employees accrue vacation on a pro-rata basis according to their scheduled hours.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 543, 'total_tokens': 636, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 44, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-caeb1096-5178-482b-8ad9-2abd9308f746-0' usage_metadata={'input_tokens': 543, 'output_tokens': 93, 'total_tokens': 636}


### Reflection

**Question 1 (office days)**
- Generated questions: useful? (did they cover "2020", "2022", "2023", "hybrid", "remote"?)
- Answer: it mention all three policy versions and say 3 days is the current one

**Question 2 (vacation days)**
- Generated questions: useful?
- Answer: it say the number is not specified.

**Overall:** MultiQueryRetriever helped on Q1, where the answer was spread over three documents and a single query could have returned only one or two. On Q2 it made no difference, since the information doesn't exist in the data. The model did not seem to be missing information on either question, and it stayed grounded in the retrieved context instead of hallucinating.

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?